# QuantClaw Harness Engineering Guide

This notebook is a beginner-friendly, end-to-end tour of the QuantClaw codebase.

The big idea is simple:

- the model is not the system
- the harness is the system around the model
- QuantClaw is one large, layered harness that lets a plain-English goal like `"go make me cash"` turn into a persistent, multi-agent, sandbox-protected, paper-trading campaign

This guide walks through every major layer of that harness — from the outer mission loop all the way down to the sandbox that runs LLM-generated factor code, the SQLite state database, the notification channels, and the Next.js dashboard.

No prior harness-engineering experience is assumed. By the end you should know *what each folder of the repo is for*, *why it exists*, and *where to read next*.

## How To Use This Notebook

Read this notebook top to bottom the first time.

As you go, pay attention to four recurring questions:

1. What job is the harness doing here?
2. Why is this logic outside the model instead of inside a prompt?
3. What state is being persisted so the system can keep going later?
4. What safety boundary stops a broad goal from turning into reckless behavior?

We will keep revisiting those questions because they are the heart of harness engineering.


## Table of Contents

**Part I — Framing**
1. What Harness Engineering Means
2. The Problem We Needed To Solve
3. The New High-Level Architecture
4. A Beginner Glossary

**Part II — The Mission Brain**
5. How A Plain-English Goal Becomes A Campaign
6. The Profit Campaign State Machine
7. OODA As The Inner Engine

**Part III — Who Does The Work**
8. The Agent Zoo: Twelve Specialists And A Base Class
9. Planner Integration And Prompt Steering
10. The LLMRouter And Sidecar: Provider Abstraction
11. The Dispatcher And The Plan DAG
12. Iteration And Evaluation Inside Each Cycle

**Part IV — Code That Gets Run**
13. Sandboxed Training And Factor Code Execution
14. The Strategy Module Contract

**Part V — The Portfolio Harness**
15. The Deployment Allocator Layer
16. The Closed-Loop Paper Deployment Runner
17. The Sentinel: Reactive Guardian

**Part VI — Memory And Observability**
18. Persistence, Recovery, And Restartability (Playbook)
19. The State Database Beyond The Playbook
20. Events, Narration, And Notification Routing

**Part VII — The Extensible Surfaces**
21. The Plugin System: Data Sources, Brokers, Engines
22. Progression, Milestones, And Onboarding
23. The Dashboard Frontend

**Part VIII — Putting It Together**
24. Safety And Risk Boundaries
25. End-To-End Walkthrough: `go make me cash`
26. Why This Is Good Harness Engineering
27. Tests As Executable Documentation
28. Practical Study Order
29. Optional Hands-On Cells
30. Final Takeaway

## 1. What Harness Engineering Means

A good beginner definition is:

> Harness engineering is the work of turning a capable model into a dependable system.

That usually means building things like:

- goal compilers
- state machines
- persistence layers
- safety rules
- evaluators
- orchestration loops
- specialized deterministic runners
- event streams and monitoring

In QuantClaw, the harness is not one file. It is a set of cooperating layers.

The important mindset shift is this:

- the LLM suggests and reasons
- the harness decides how the system behaves over time

That is why this project now treats a broad request like `"go make me cash"` as a persistent campaign, not as a one-shot prompt.


## 2. The Problem We Needed To Solve

Before the redesign, QuantClaw already had a useful OODA loop and planning system. But broad profit-seeking goals still had a few important limitations.

The old shape was closer to this:

- accept a goal
- plan a cycle
- run agents
- evaluate the result
- maybe iterate a few times
- eventually stop

That is helpful, but it is not yet a real long-lived profit-seeking harness.

What was missing:

- no persistent outer campaign for broad goals
- no portfolio memory of incumbents versus challengers
- no allocator that could keep multiple paper strategies alive at once
- no deterministic runner that continuously executed the active paper portfolio
- no strong restart story for an in-progress broad campaign

So the redesign moved recursion and persistence upward.

Instead of letting the system think in isolated cycles, we gave it a standing mission layer above those cycles.


## 3. The New High-Level Architecture

Here is the fuller mental model of the current system:

```text
                  User / Dashboard / API / Daemon
                                 |
                                 v
                     Broad goal arrives (set_goal)
                                 |
                                 v
      +-------------- CampaignManager --------------+
      | broad-goal matcher → ProfitCampaign object  |
      | phases: discover → validate → paper → refine |
      +---------------------+-----------------------+
                            |
                            v
                        OODALoop
        (observe → orient → decide → act → learn)
                            |
            +---------------+---------------+
            |               |               |
            v               v               v
        Planner        Dispatcher      Evaluator
       (LLMRouter →   (DAG of plan     (scoring,
        provider +     steps, parallel  divergence
        sidecar)       where possible)  calibration)
            |
            v
     +--- Agent Zoo (12 specialists) ---+
     | Researcher  Miner     Trainer    |
     | Ingestor    Validator  Executor  |
     | Reporter    Risk       Sentinel  |
     | Compliance  Debugger   Scheduler |
     +-----------------+----------------+
                       |
         +-------------+--------------+
         |                            |
         v                            v
      Sandbox                    DeploymentAllocator
  (subprocess isolation,        (portfolio of paper
   resource limits, AST          strategies: active /
   validation, env stripping)    watchlist / retired)
                                       |
                                       v
                                   Executor
                            (run_deployments task →
                             load strategies, fetch
                             data, aggregate weights,
                             submit paper orders)

Cross-cutting layers (used by everything above):

  Playbook (JSONL)     ← campaign_state, deployment_state, allocation_decision
  State DB (SQLite)    ← tasks, events, plans, sessions
  EventBus             ← 50+ typed events, routed to notification sinks
  Notifications        ← Telegram, Discord, Slack (urgency-routed)
  Plugin Manager       ← data sources, brokers, engines, asset universes
  Dashboard (Next.js)  ← trading floor + plan approvals + stop button
```

The most important design choice is this:

- **OODA** is the inner engine
- **CampaignManager** is the outer mission harness
- **DeploymentAllocator** is the portfolio harness
- **Executor + Sandbox** is the deterministic execution harness
- **Playbook + State DB** is the memory harness
- **EventBus + Notifications** is the observability harness
- **Plugins** is the extensibility harness

Each of the sections below is a deep dive into one layer.

## 4. A Beginner Glossary

A few terms will show up repeatedly:

- **Harness**: the surrounding system that controls behavior, safety, memory, execution, and iteration.
- **Campaign**: a long-lived mission created from a broad goal.
- **OODA**: Observe, Orient, Decide, Act — the inner cycle engine.
- **Held-out evaluation**: testing on data that was not used during training or selection.
- **Paper trading**: simulated trading with realistic portfolio updates but without risking real money.
- **Allocator**: a layer that decides which candidate strategies deserve active capital slots.
- **Playbook**: append-only JSONL memory store in `data/playbook.jsonl`.
- **State DB**: SQLite database at `data/quantclaw.db` for structured state (tasks, events, plans, sessions).
- **Sandbox**: isolated subprocess with resource limits used to run LLM-generated factor/model code.
- **Broad goal**: a plain-English mission like `"go make me cash"` or `"find profitable strategies"`.
- **Deterministic runner**: logic that follows a fixed program instead of asking the planner to invent the next step.
- **Agent**: a specialist class in `quantclaw/agents/` with a scoped job (miner, trainer, executor, etc.).
- **Plan / PlanStep**: a DAG of steps (each assigned to an agent) produced by the Planner and executed by the Dispatcher.
- **LLMRouter**: the provider-abstraction layer that lets the same agent code run on Anthropic, OpenAI, or Ollama.
- **Sidecar**: a small Node.js server that proxies OAuth-authenticated LLM calls so the Python side never handles OAuth tokens directly.
- **Event**: a typed `Event(type, payload, source_agent)` published on the EventBus — consumed by Sentinel, dashboards, and notification sinks.
- **Notification sink**: a channel like Telegram, Discord, or Slack that events get routed to based on urgency rules.
- **Plugin**: a pluggable module registered with the PluginManager — data source, broker, engine, or asset universe.
- **Level / Milestone**: progression markers (Observer → Architect) that gate which agents and strategy templates are unlocked.
- **Strategy module**: a Python file with a `Strategy` class implementing `signals(data)` and `allocate(scores, portfolio)`.

If you keep these in mind, the rest of the notebook becomes much easier to follow.

## 5. How A Plain-English Goal Becomes A Campaign

The first big harness upgrade lives in `quantclaw/orchestration/campaigns.py`.

The `CampaignManager` has one crucial responsibility:

- recognize when a goal is broad enough to require a persistent campaign
- activate a `ProfitCampaign`
- generate the next subgoal for the current phase
- persist the campaign so it can survive restarts

Broad goals are matched by patterns in `quantclaw/config/default.yaml`, such as:

- `make money`
- `make me cash`
- `go make money`
- `go make me cash`
- `find alpha`
- `grow capital`

That means the phrase `"go make me cash"` is not treated as a literal trading instruction. It is compiled into a campaign objective.

Adapted example:

```python
manager = CampaignManager(playbook, config)
campaign = manager.activate("go make me cash")

if campaign:
    subgoal = manager.next_subgoal(campaign)
    # "Explore for profitable trading strategies..."
```

Why this is good harness engineering:

- natural language is converted into structured runtime state
- the system can tell the difference between a narrow task and an open-ended mission
- the model no longer has to remember the mission structure by itself


## 6. The Profit Campaign State Machine

The `ProfitCampaign` object is the heart of the outer harness.

It stores fields like:

- `root_goal`
- `phase`
- `total_cycles`
- `phase_cycles`
- `validated_candidates`
- `paper_deployments`
- `best_sharpe`
- `best_held_out_sharpe`
- `last_subgoal`
- `last_verdict`
- `paper_only`

The phase machine is intentionally explicit:

- `discover`
- `validate`
- `paper`
- `refine`

That phase split matters because each phase has a different job:

- **discover**: search broadly across factor families, universes, and model types
- **validate**: run stricter held-out checks and reject overfit ideas
- **paper**: manage a portfolio of paper deployments and compare live paper behavior with expectations
- **refine**: deliberately change hypothesis families when the search path stalls

The campaign also has promotion thresholds in config:

```yaml
campaigns:
  min_discovery_cycles: 2
  discovery_promote_sharpe: 0.75
  validation_promote_held_out_sharpe: 0.25
  max_consecutive_failures: 2
```

Adapted transition logic:

```python
if campaign.phase in (DISCOVER, VALIDATE):
    if metrics["validated"] or best_held_out >= validate_threshold:
        campaign.phase = PAPER
    elif phase == DISCOVER and phase_cycles >= min_discovery_cycles and best_sharpe >= discover_threshold:
        campaign.phase = VALIDATE
    elif consecutive_failures >= max_failures:
        campaign.phase = REFINE
elif campaign.phase == PAPER and consecutive_failures >= max_failures:
    campaign.phase = REFINE
elif campaign.phase == REFINE and verdict in ("iterate", "pursue"):
    campaign.phase = DISCOVER
```

This is a classic harness pattern: make progression explicit, testable, and recoverable.


## 7. OODA As The Inner Engine

QuantClaw did not throw away OODA. It made OODA more useful by wrapping it in stronger mission logic.

The main file is `quantclaw/orchestration/ooda.py`.

The key idea is:

- the campaign decides the mission phase and the next subgoal
- OODA decides how to execute the next cycle

In the current design:

- `observe()` now includes campaign and deployment context
- `orient()` asks the campaign manager for the next subgoal
- `decide()` can choose between normal planning and a deterministic paper runner plan
- `run_cycle()` records campaign progress and allocator decisions after evaluation
- `run_continuous()` keeps broad campaigns going instead of stopping after a small fixed number of cycles

Adapted example:

```python
state = await ooda.observe()
orientation = await ooda.orient(state)
plan = await ooda.decide(orientation)
results = await ooda.act(plan)
evaluation = await ooda._evaluate_results(results, iteration=1)

await campaign_manager.record_cycle(campaign, results, evaluation)
await deployment_allocator.rebalance(campaign.id, cycle_no, results, evaluation)
```

This is one of the best design moves in the project because it preserves the existing orchestration engine while giving it a long-lived mission brain above it.


## 8. The Agent Zoo: Twelve Specialists And A Base Class

Up to now we have talked about "the harness runs agents" without saying *what* agents. Let us fix that.

Every agent lives in [quantclaw/agents/](quantclaw/agents/) and extends `BaseAgent` from [quantclaw/agents/base.py](quantclaw/agents/base.py). The base class provides:

- `name`, `model`, `daemon`, `max_retries` fields
- `execute(task)`, `verify(result)`, `plan(goal)`, `on_event(event)` hook methods
- a run-loop that emits `agent.task_started`, `agent.task_completed`, and `agent.task_failed` events

The full roster (single source of truth in [quantclaw/agents/manifest.py](quantclaw/agents/manifest.py)):

| Agent | File | LLM? | Daemon? | Job |
| --- | --- | --- | --- | --- |
| Researcher | [researcher.py](quantclaw/agents/researcher.py) | yes | no | LLM-driven web search + factor hypothesis discovery |
| Ingestor | [ingestor.py](quantclaw/agents/ingestor.py) | no | no | Fetches OHLCV via data plugins |
| Miner | [miner.py](quantclaw/agents/miner.py) | yes | no | Evolutionary factor discovery — generates, tests in sandbox, mutates top performers |
| Trainer | [trainer.py](quantclaw/agents/trainer.py) | yes | no | Trains ML models (sklearn, xgboost, lightgbm) in sandbox; emits Strategy class code |
| Validator | [validator.py](quantclaw/agents/validator.py) | no | no | Replays strategy in sandbox (task=`backtest`) + runs held-out validation (task=`validate`) in one agent |
| Executor | [executor.py](quantclaw/agents/executor.py) | no | no | Paper/live order submission; supports `run_deployments` |
| Risk Monitor | [risk_monitor.py](quantclaw/agents/risk_monitor.py) | no | yes | Daemon — drawdown, sector concentration, leverage checks |
| Compliance | [compliance.py](quantclaw/agents/compliance.py) | no | no | Rule-based trade gatekeeper |
| Sentinel | [sentinel.py](quantclaw/agents/sentinel.py) | no | yes | Event listener — fires alerts on failure streaks, cost spikes, regime changes |
| Debugger | [debugger.py](quantclaw/agents/debugger.py) | yes | no | LLM diagnostic invoked only on failure with error context |
| Scheduler | [scheduler.py](quantclaw/agents/scheduler.py) | no | yes | Cron-style trigger daemon |
| Reporter | [reporter.py](quantclaw/agents/reporter.py) | yes | no | LLM executive summary; always last step in a pipeline |

Two pedagogical points:

1. **Not every agent uses an LLM.** Ingestor, validator, risk_monitor, compliance, and scheduler are deterministic code. This is harness engineering: the LLM is a *tool inside* the system, not the whole system. When determinism is appropriate, use it.

2. **The manifest is the API surface.** [manifest.py](quantclaw/agents/manifest.py) stores an `AgentSpec` per agent with `inputs`, `outputs`, `depends_on`, `feeds_into`, and `task_schema`. The Planner reads this manifest to know which agent to call for which job, which agents need which upstream outputs, and what task shape each expects. Adding a new agent means adding a class *and* a manifest entry — both or neither.

**On the Validator merge**: the roster used to be thirteen, with separate `Backtester` (in-sample replay) and `Evaluator` (held-out validation). They were two halves of the same job — replay a strategy against market data — run on different windows. Merging them into one `Validator` with two task modes (`backtest` and `validate`) collapses an unnecessary agent boundary without losing the held-out-is-sacred distinction: task mode makes the distinction explicit on each invocation.

Adjacent to the agents, [quantclaw/agents/tools/](quantclaw/agents/tools/) holds shared capabilities (for example `web_search.py`) that agents call into — think of them as skills, not agents.

## 9. Planner Integration And Prompt Steering

The planner still matters a lot. The difference is that it is now being fed much better context.

The main file is [quantclaw/execution/planner.py](quantclaw/execution/planner.py).

The planner prompt now receives structured sections such as:

- recent playbook context
- campaign context
- deployment allocator context
- prior iteration failures and things to avoid
- exploration mode and temperature
- the **agent manifest** (so the planner knows every agent's inputs, outputs, and dependencies)

That means planning is no longer stateless. The LLM sees where the mission currently is.

Two especially strong planner rules were added:

- broad goals like `make money` must plan a full pipeline, not a lazy one-step answer
- if campaign phase is `paper` and there are active deployments, do not suggest live trading

A simplified version of the planner steering looks like this:

```python
context = {
    "campaign_context": campaign.to_prompt_context(),
    "deployment_context": deployment_context,
    "playbook_context": recent_entries,
    "iteration_context": prior_failures,
    "agent_manifest": format_manifest_for_prompt(),
}
plan = await planner.create_plan(request=subgoal, context=context)
```

The planner returns a `Plan` (see [quantclaw/execution/plan.py](quantclaw/execution/plan.py)), which is a DAG of `PlanStep` objects each with `id`, `agent`, `task`, `description`, `depends_on`, and `status`. That shape is critical — it is the contract between *thinking* and *doing*.

Why this matters:

- prompt engineering becomes harness engineering when the prompt is assembled from real runtime state
- the model is guided by mission phase, portfolio state, and prior failures
- the planner is creative, but the harness constrains what kind of creativity is useful

## 10. The LLMRouter And Sidecar: Provider Abstraction

Notice something subtle about the agents: they all say "call the LLM," but none of them says *which* LLM. That is intentional. Provider selection is a harness concern, not an agent concern.

### 10.1 The LLMRouter

The routing lives in [quantclaw/execution/router.py](quantclaw/execution/router.py).

The `LLMRouter` class exposes four key methods:

```python
router.get_model("miner")        # -> "gpt"
router.get_provider("miner")     # -> {"provider": "openai", "model": "gpt-4o"}
router.get_temperature("miner")  # -> 0.9 (config override > default)
await router.call("miner", messages=[...], system="...", temperature=0.9)
```

Per-agent model assignment comes from [quantclaw/config/default.yaml](quantclaw/config/default.yaml) — the `models`, `providers`, and `temperatures` blocks. Reading that config is the fastest way to understand the system's LLM strategy: opus for deep reasoning (executor, risk_monitor, planner, backtester), sonnet for summaries (reporter, sentinel), gpt for creative exploration (miner), and local Ollama as a fallback.

### 10.2 Two paths: API key vs OAuth

Inside `router.call()` there are two branches:

1. **API-key path** — direct Anthropic / OpenAI / Ollama client calls using `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` from the environment.
2. **OAuth path** — if an OAuth token is present (e.g. the user is logged in via Claude Code or Codex), the router forwards the call to a local Node.js sidecar at `http://localhost:24122`.

Why a sidecar? Because Anthropic and OpenAI OAuth flows are implemented in their official CLIs using beta headers and custom token refresh logic. Reproducing that in Python correctly is error-prone. It is far more reliable to use the same SDKs the official CLIs use — which happen to be JavaScript — and keep the Python side OAuth-unaware.

### 10.3 The sidecar

See [quantclaw/sidecar/server.js](quantclaw/sidecar/server.js). Three endpoint handlers:

- `handleOpenAI` — forwards to ChatGPT backend with Responses API using the user's OAuth access token
- `handleAnthropic` — uses `@anthropic-ai/sdk` with the `claude-code-20250219,oauth-2025-04-20` beta headers
- `handleGoogle` — loads Gemini CLI OAuth credentials from `~/.gemini/oauth_creds.json` and calls the google-genai SDK

The sidecar reads credentials from `data/oauth_credentials.json` (written by [quantclaw/dashboard/oauth.py](quantclaw/dashboard/oauth.py) during onboarding) and never exposes them back to the Python process.

### 10.4 Cost tracking and budget warnings

Because the router is the single choke point for every LLM call, it is also the right place to track spend. `LLMRouter` owns a `CostTracker` that:

- records input / output tokens per call
- maintains USD totals per-agent and per-model, using a rates table from [config/default.yaml](quantclaw/config/default.yaml) under `cost.rates`
- fires a `cost.budget_warning` event (Section 20) the first time spend crosses each configured threshold — default `[0.5, 0.8, 1.0]` of `budget_usd`
- exposes `router.cost.summary()` for the dashboard

```yaml
# Minimal config
cost:
  budget_usd: 10.0
  warning_thresholds: [0.5, 0.8, 1.0]
  rates:
    claude-opus:   [15.00, 75.00]   # per 1M [input, output]
    claude-sonnet: [3.00, 15.00]
    gpt-4o:        [2.50, 10.00]
```

Caveat: the sidecar does not yet surface `usage` from provider responses, so OAuth-path calls currently count at $0. API-key-path calls and Ollama calls are accounted correctly.

### 10.5 Why this is good harness engineering

- Agents depend on a *capability* ("call an LLM"), not a *vendor*
- Changing from Anthropic to Ollama is a config edit, not a code change
- OAuth complexity is isolated behind a single process boundary
- Cost tracking and rate limiting have exactly one place to live — and now the router actually does that

## 11. The Dispatcher And The Plan DAG

The Planner produces a `Plan`. The `Dispatcher` *runs* it. That separation is the second critical seam in this architecture.

The dispatcher lives in [quantclaw/execution/dispatcher.py](quantclaw/execution/dispatcher.py). Its core responsibility is: **walk the plan DAG, respect dependencies, parallelize where allowed, and inject upstream results into downstream tasks.**

Pseudocode of the execution loop:

```python
async def execute_plan(plan):
    while not plan.is_complete():
        ready_steps = plan.get_ready_steps()   # no unmet deps, approved, pending
        if not ready_steps:
            break
        results = await dispatch_parallel([
            (step, build_task_with_upstream(step, plan))
            for step in ready_steps
        ])
        for step, result in zip(ready_steps, results):
            step.status = COMPLETED if result.ok else FAILED
            plan.results[step.id] = result
```

The subtle pieces:

- **Upstream injection** — before dispatching step `s`, the dispatcher reads `s.depends_on`, pulls the successful results of those prior steps from `plan.results`, and merges them into the outgoing task under key `_upstream_results`. This is how the trainer gets the miner's factors without the planner having to hand-wire them.
- **Failure propagation** — if step A fails, all steps that depend on A are marked `SKIPPED` rather than failed. This makes post-hoc analysis sane: you can tell "nothing ran" apart from "it tried and broke."
- **Events** — the dispatcher emits `orchestration.step_started`, `orchestration.step_completed`, `orchestration.step_failed`, and `orchestration.broadcast` (when multiple steps run in parallel). Those events are what drives dashboard progress bars and Sentinel's failure-streak counter.
- **Cancellation** — an `_cancel` event is checked before each round, so the dashboard stop button can actually stop a cycle mid-plan without killing the whole process.

The companion classes are worth knowing:

- [`AgentPool`](quantclaw/execution/pool.py) — a registry of instantiated agents keyed by name. The dispatcher calls `pool.get(step.agent).run(task)`.
- [`ToolLoop`](quantclaw/execution/tool_loop.py) — a provider-agnostic tool-use loop for when an agent needs to call tools like `web_search` iteratively (max 8 rounds). Supports both Anthropic Messages tool-use shape and OpenAI Chat Completions tool-use shape.
- [`Plan` / `PlanStep`](quantclaw/execution/plan.py) — the approval machinery. `approve_all()`, `approve_step()`, `skip_step()`, `reject()`. Chat-triggered plans auto-approve (see the `e9b53dc` commit). Manual plans require explicit user approval in the dashboard.

This is a clean example of the "thinking / doing / tracking" tripartite split: planner thinks, dispatcher does, plan-store tracks.

## 12. Iteration And Evaluation Inside Each Cycle

There are actually two loops now:

- an outer campaign loop across many cycles
- an inner evaluation loop inside each cycle

Inside `OODALoop.run_cycle()`, QuantClaw can iterate up to `max_iterations_per_cycle` times (default 3, set in [config/default.yaml](quantclaw/config/default.yaml)).

The flow is roughly:

1. plan a workflow
2. dispatch it through the DAG
3. evaluate the results
4. if needed, refine the next iteration using diagnostics

There are some good harness details here:

- zero-trade results are treated as a pipeline problem, not instantly as a strategy failure
- the system logs evaluation divergences between the agent's self-reported score and the evaluator's re-score, which feeds a calibration table over time
- exploration temperature shifts toward exploitation during refinement (see the `exploration` config block)
- the iteration context explicitly tells the planner what not to repeat
- the Debugger agent only runs on failure, with the exact error context from the dispatcher's `_upstream_results`

This is another excellent example of harness engineering: the model is allowed to try again, but the system forces it to learn from the previous attempt.

## 13. Sandboxed Training And Factor Code Execution

Now for one of the most important safety layers in the whole system.

Miner and Trainer agents generate Python code — factor expressions, ML training scripts, feature pipelines. That code is, by definition, untrusted: an LLM produced it. Running it directly in the main process would be reckless. Running it in a container would be correct but heavy. QuantClaw splits the difference with a subprocess-level sandbox.

### 13.1 The Sandbox class

See [quantclaw/sandbox/sandbox.py](quantclaw/sandbox/sandbox.py).

```python
sandbox = Sandbox(config)
result = await sandbox.execute_code(
    code=llm_generated_code,
    timeout=60,                # seconds
    data={"prices": df},       # written to parquet for code to read
)
# result: SandboxResult(status, stdout, stderr, result, import_warnings)
```

The key protections:

- **Subprocess isolation** — `sys.executable` spawns a fresh Python process, so a segfault or runaway loop cannot take down the main service
- **Resource caps** — `timeout` (60s default), `max_memory_mb` (512 default), `max_output` (100 KB), `max_concurrent` (3) — all live in the `sandbox:` block of [config/default.yaml](quantclaw/config/default.yaml)
- **AST validation (enforced, not advisory)** — before spawning, the code is parsed and rejected if any check fires critical
- **Data hand-off by file** — the main process writes dataframes to parquet files; the sandbox reads them. No shared memory, no pickle attack surface.

### 13.2 The security layer

See [quantclaw/sandbox/security.py](quantclaw/sandbox/security.py). `validate_imports()` walks the AST and produces `ImportWarning` records with a `critical` flag. The Sandbox **blocks execution if any warning is critical**.

```python
ALLOWED_IMPORTS = {
    # Stdlib basics
    "math", "statistics", "datetime", "time", "json", "re",
    "collections", "itertools", "functools", "decimal", "fractions",
    "typing", "warnings", "os", "sys", "pathlib",
    # Scientific stack
    "numpy", "pandas", "scipy", "sklearn", "statsmodels", "joblib",
    "xgboost", "lightgbm",
    "np", "pd",
    # Own modules (sandbox PYTHONPATH exposes them)
    "quantclaw",
}

BLOCKED_BUILTINS = {
    "__import__", "exec", "eval", "compile",
    "getattr", "setattr", "delattr",
    "globals", "locals", "vars",
}

BLOCKED_DUNDER_ATTRS = {
    "__class__", "__bases__", "__mro__", "__subclasses__",
    "__globals__", "__builtins__", "__import__",
    "__dict__", "__getattribute__", "__getattr__",
    "__code__", "__closure__",
}
```

Three classes of check, all emitted as AST-walk findings:

1. **Import whitelist (critical)** — any `import X` or `from X import Y` where `X` is not in `ALLOWED_IMPORTS` is a critical warning and blocks execution. So `import socket`, `import subprocess`, `from urllib.request import urlopen` all fail *before* the subprocess is spawned. Network and shell escape hatches are simply unavailable.

2. **Blocked builtins (critical)** — direct calls to `exec()`, `eval()`, `compile()`, `__import__()`, `getattr()`, `setattr()`, `delattr()`, `globals()`, `locals()`, `vars()`. These are the primary dynamic-evaluation and reflection primitives that let sandboxed code rewrite itself or probe the interpreter.

3. **Dunder attribute access (critical)** — expressions like `().__class__.__mro__[1].__subclasses__()` are the canonical Python sandbox escape. Walking up `__class__` and `__subclasses__` from any object reaches `os` even if you cannot `import os`. Blocking attribute access to the dunder names on the list closes that path.

Plus environment stripping:

```python
SENSITIVE_ENV_PATTERNS = {
    "KEY", "TOKEN", "SECRET", "PASSWORD", "CREDENTIAL", "AUTH",
    "API", "PRIVATE", "CERT", "COOKIE", "SESSION", "OAUTH",
}
```

`strip_env()` removes any env var whose name contains one of those patterns before handing the env to the subprocess. Even if the sandboxed code reads `os.environ` (which it can, since `os` is allowed), it cannot exfiltrate your API keys — they are not there.

### 13.3 Factor evaluator and model trainer

Two layers sit on top of the generic sandbox:

- [quantclaw/sandbox/factor_evaluator.py](quantclaw/sandbox/factor_evaluator.py) — the Miner's harness. Takes a factor expression, wraps it in a scoring template, runs in sandbox, returns IC / Sharpe.
- [quantclaw/sandbox/model_trainer.py](quantclaw/sandbox/model_trainer.py) — the Trainer's harness. Supports `gradient_boosting`, `xgboost`, `lightgbm`, `random_forest`, `logistic_regression`, `lstm`, `transformer`. Wraps the training call, serializes model to disk, returns paths and metrics.

### 13.4 Defense in depth

The pedagogical takeaway: **"run LLM-generated code" is one of the most dangerous verbs in software engineering.** QuantClaw takes it seriously by layering four independent defenses:

1. AST pre-flight (rejects dangerous imports, builtins, dunder attrs *before* any code runs)
2. Env stripping (no secrets in the subprocess)
3. Subprocess isolation (crashes and runaway loops stay contained)
4. Resource caps (timeout, memory, output, concurrency)

Any single layer is breakable. All four together would require a meaningful 0-day.

## 14. The Strategy Module Contract

So far we have been vague about "a strategy." Let us pin it down.

A strategy in QuantClaw is a plain Python module containing a class named `Strategy` that implements a tiny interface. That interface is the contract between the *search* side of the system (miner, trainer, backtester) and the *execution* side (executor, allocator).

### 14.1 The contract

From [quantclaw/strategy/loader.py](quantclaw/strategy/loader.py) and [quantclaw/strategy/generator.py](quantclaw/strategy/generator.py):

```python
class Strategy:
    name: str                      # optional, for logs
    description: str               # optional
    universe: list[str]            # tickers this strategy trades
    frequency: str                 # "daily" | "weekly" | "monthly"

    def signals(self, data) -> dict[str, float]:
        """Return a score per symbol (higher = more bullish)."""
        ...

    def allocate(self, scores, portfolio) -> dict[str, float]:
        """Return target portfolio weights (sum to 1 for long-only)."""
        ...

    def risk_check(self, orders, portfolio) -> bool:
        """Optional: veto an order batch."""
        return True
```

Two methods, two optional hooks. That is the whole contract.

Because the contract is narrow, the system can treat strategies as interchangeable units. The allocator does not care whether a strategy came from a template, a miner run, or a human — if it has `signals()` and `allocate()`, it fits into the portfolio.

### 14.2 The template library

Templates live in [quantclaw/strategy/templates/](quantclaw/strategy/templates/), organized by difficulty:

- **baselines/** — `equal_weight`, `random_picks`, `spy_benchmark` (sanity checks; every new strategy should beat these)
- **beginner/** — `buy_and_hold`, `moving_average`, `momentum`, `mean_reversion`
- **intermediate/** — `pairs`, `risk_parity`, `sector_rotation`
- **advanced/** — `ml_signal`
- **options/** — `wheel`

Progression (Section 22) unlocks templates by level. Observers start with baselines only; Quants unlock advanced + options.

### 14.3 The supporting cast

- [quantclaw/strategy/loader.py](quantclaw/strategy/loader.py) — imports a `.py` file, validates `class Strategy` exists, instantiates it
- [quantclaw/strategy/generator.py](quantclaw/strategy/generator.py) — LLM-powered strategy code generation; validates the contract before saving
- [quantclaw/strategy/audit.py](quantclaw/strategy/audit.py) — logs every signal, allocation, risk_check, trade, rebalance, and skip event per date; CSV-exportable
- [quantclaw/strategy/runner.py](quantclaw/strategy/runner.py) — loads a strategy, fetches data via plugin, and calls the engine's `backtest()`

### 14.4 Why this is good harness engineering

- The contract is **simple enough for an LLM to produce reliably** but **rich enough to back a real portfolio**
- The `Strategy` class is **code, not prompts** — once generated, it is deterministic, reproducible, and auditable
- The audit log means every decision a strategy made can be replayed and investigated

## 15. The Deployment Allocator Layer

The next major upgrade lives in [quantclaw/orchestration/deployments.py](quantclaw/orchestration/deployments.py).

This file introduces two important ideas:

- a `PaperDeployment` record for each candidate strategy
- a `DeploymentAllocator` that decides which candidates are active, watchlisted, or retired

This is the shift from **single-winner thinking** to **portfolio thinking**.

Each deployment tracks:

- `strategy_path`, `score`, `sharpe`, `held_out_sharpe`
- `annual_return`, `max_drawdown`
- `paper_runs`, `last_equity`, `allocation_pct`
- `status` = active, watchlist, or retired

Qualification rules are deliberately explicit:

```python
def _qualifies(verdict, held_out, sharpe):
    if verdict == "validated":
        return True
    if held_out >= min_held_out:
        return True
    return held_out >= watchlist_min_held_out
```

The score function is also explicit and readable:

```python
score = (
    held_out_sharpe * 0.6
    + sharpe * 0.25
    + annual_return * 0.15 * 10
    - abs(max_drawdown) * 0.2 * 10
)
```

Then the allocator does something very practical:

- sort all candidates by score
- keep the top `max_active_paper_deployments` as active
- keep the next `paper_watchlist_size` as watchlist
- retire the rest
- normalize active allocation weights so they sum to 100%

That means the system can now say:

- these two strategies deserve active paper slots
- these two others are worth watching
- the rest are not good enough right now

This is one of the most important harness upgrades in the whole project.

## 16. The Closed-Loop Paper Deployment Runner

The allocator alone is not enough. It can choose active strategies, but something still needs to run them.

That missing last mile now lives in [quantclaw/agents/executor.py](quantclaw/agents/executor.py).

The executor gained a new task type:

- `run_deployments`

This is what makes the system a true closed-loop paper campaign rather than just a ranking engine.

At a high level, `_run_paper_deployments()` does this:

1. read active deployments from the task
2. load each strategy module from `strategy_path` via the Strategy loader (Section 14)
3. fetch recent market data for that strategy's universe through the data plugin
4. call `signals()` and `allocate()` on the strategy
5. scale the target weights by the allocator's `allocation_pct`
6. aggregate all target weights into one paper portfolio target
7. build rebalance orders
8. submit those orders through paper execution
9. return deployment-level updates plus the combined paper portfolio state

Adapted example:

```python
for deployment in active_deployments:
    strategy = load_strategy(deployment["strategy_path"])
    data = load_market_data(symbols, start, end)
    signals = strategy.signals(data_proxy)
    target_weights = strategy.allocate(signals, portfolio_proxy)

    scaled = scale_by_allocation_pct(target_weights, deployment["allocation_pct"])
    aggregate_weights = merge(aggregate_weights, scaled)

orders = build_orders_from_target_weights(aggregate_weights, latest_prices)
result = await execute_paper(orders)
```

The really nice engineering choice here is that paper-phase execution is no longer dependent on the planner being clever every single cycle. The system has a dedicated deterministic runner for a well-defined job.

## 17. The Sentinel: Reactive Guardian

If the Allocator is the *proactive* portfolio layer (pick winners), the Sentinel is the *reactive* safety layer (notice trouble).

See [quantclaw/agents/sentinel.py](quantclaw/agents/sentinel.py). It is a `daemon=True` agent, which means it never shows up in a plan — it runs in the background and reacts to events.

### 17.1 What it listens for

Sentinel subscribes to the EventBus and pattern-matches events against a rules table:

| Rule | Trigger | Severity |
| --- | --- | --- |
| `agent_failure_streak` | same agent fails 3 times in a row | high |
| `drawdown_warning` | equity drops to 80% of max drawdown tolerance | critical |
| `trade_failure` | `trade.reconciliation_fail` | critical |
| `cost_warning` | `cost.budget_warning` | high |
| `market_gap` | `market.gap_detected` | high |
| `regime_change` | `market.regime_change` | normal |

### 17.2 What it does

On a matching event Sentinel emits an alert event, which the notification sinks (Section 20) pick up and send to Telegram / Discord / Slack according to the routing config.

Internally it keeps three pieces of state:

- `_failure_counts: dict[agent_name, int]` — incremented on `agent.task_failed`, reset on `agent.task_completed`
- `_alerts_fired: list` — audit trail of alerts this session
- `_rules: dict` — the rules table, loaded once from config

### 17.3 Why a separate agent?

The critical design point: Sentinel is **reactive**, not **planned**. The planner never schedules a Sentinel step. Sentinel listens to every event the system produces and fires alerts on its own clock.

This matters because:

- reactive safety should not depend on the planner remembering to schedule it
- the guardian must survive plan failures (imagine Sentinel only ran after a successful plan — that is exactly the wrong shape)
- daemon agents are a different architectural primitive than pipeline agents, and this codebase makes the distinction explicit

Compare this to RiskMonitor ([risk_monitor.py](quantclaw/agents/risk_monitor.py)), which is also a daemon but watches *portfolio state* rather than *event stream*. Both are reactive; they just watch different inputs.

## 18. Persistence, Recovery, And Restartability (The Playbook)

A serious harness cannot depend on in-memory luck. It needs durable state.

QuantClaw uses the Playbook for that. The main file is [quantclaw/orchestration/playbook.py](quantclaw/orchestration/playbook.py).

The Playbook is **append-only JSONL** storage at `data/playbook.jsonl`. That is a simple design, but a very good one: append-only means no corruption from concurrent writes, every entry is a timestamped record, and the whole file is inspectable with `cat` and `jq`.

Entry types currently include:

- `campaign_state` — snapshot of a ProfitCampaign (phase, metrics, cycle counts)
- `deployment_state` — snapshot of a single PaperDeployment
- `allocation_decision` — a summary of each allocator rebalance
- `ceo_preference` — user-stated preferences that should survive restarts
- `evaluator_divergence` / `evaluator_calibration` — cycle evaluation divergences
- `strategy_result`, `what_failed`, `factor_library`, `trust_milestone`, `agent_performance`, `market_observation`, `scaffolding_experiment` — free-form records

### 18.1 Startup restore

Startup paths were updated to restore state:

- [quantclaw/dashboard/api.py](quantclaw/dashboard/api.py) calls `await ooda.restore_persistent_state()`
- [quantclaw/daemon.py](quantclaw/daemon.py) also restores persistent state on startup

That means:

- if the backend restarts, the standing campaign can resume
- if the daemon restarts, active paper portfolio context is not lost
- state is inspectable because it lives in explicit records, not hidden model memory

### 18.2 Why append-only?

A mutable state store would need locking, migrations, and rollback logic. An append-only log needs none of those. To "update" a deployment, you append a new `deployment_state` record; to read the current state, you fold the log forward. The file is trivially tailable, greppable, and forensic-friendly.

### 18.3 Compaction

An unbounded append-only log is not a good idea forever. Every `restore_persistent_state()` re-reads the whole file, so startup latency grows linearly. The Playbook handles this with scheduled compaction:

- every `compact_check_every` writes (default 500), the Playbook checks if the live file exceeds `max_file_bytes` (default 20 MB) or `max_entries` (default 5000)
- if so, it **archives** the full current file to `data/playbook.jsonl.archive.YYYYMMDD-HHMMSS.gz` (gzip-compressed, full history preserved)
- then it **rewrites** the live file retaining only the entries worth loading hot:
  - for `campaign_state` and `deployment_state`: keep only the **latest snapshot per logical key** (campaign_id / deployment_id / strategy_path) — older snapshots are superseded
  - for everything else: keep the **tail** of most recent entries up to the entry budget
- the rewrite is atomic (write `.jsonl.tmp`, then `os.replace()`)

Pedagogical point: this is the classic log-compaction pattern (Kafka / LSM-tree flavour), scaled down to a single JSONL file. The system keeps full history in archives for forensics and replay, while the hot path reads only the entries that matter.

### 18.4 Why this is good harness engineering

- **Prefer durable logs over mutable state** — Playbook writes never overwrite
- **Hot path is bounded** — compaction keeps startup fast even after months of runs
- **History is cheap** — gzipped archives keep the full record for replay at near-zero storage cost
- **Snapshots dedupe automatically** — the latest-per-key rule means a campaign with 10,000 cycles does not leave 10,000 `campaign_state` entries in the live log

## 19. The State Database Beyond The Playbook

The Playbook is an excellent mission log, but some state wants to be queryable — "show me all failed tasks in the last hour," "list plans by status," "count events by type." For that QuantClaw has a SQLite database at `data/quantclaw.db`, managed by [quantclaw/state/](quantclaw/state/).

### 19.1 Schema

See [quantclaw/state/db.py](quantclaw/state/db.py). Four tables:

```sql
CREATE TABLE tasks (
    id TEXT PRIMARY KEY,
    agent TEXT,
    command TEXT,
    status TEXT,
    result TEXT,     -- JSON
    error TEXT,
    created_at TEXT,
    started_at TEXT,
    completed_at TEXT
);

CREATE TABLE events (
    id TEXT PRIMARY KEY,
    event_type TEXT,
    payload TEXT,    -- JSON
    source_agent TEXT,
    created_at TEXT
);

CREATE TABLE plans (
    id TEXT PRIMARY KEY,
    description TEXT,
    steps_json TEXT,
    status TEXT,
    created_at TEXT,
    updated_at TEXT
);

CREATE TABLE sessions (
    id TEXT PRIMARY KEY,
    session_type TEXT,
    started_at TEXT,
    ended_at TEXT,
    metadata TEXT
);
```

### 19.2 How it is used

- [quantclaw/state/event_persister.py](quantclaw/state/event_persister.py) — subscribes to the EventBus and batch-writes events every 5 seconds. This keeps event volume off the hot path.
- [quantclaw/state/plans.py](quantclaw/state/plans.py) — `PlanStore` with `save()`, `get()`, `list_by_status()`, `update_status()`. The dashboard reads from here to show in-flight plans.
- [quantclaw/state/tasks.py](quantclaw/state/tasks.py) — task history per agent; powers the dashboard Logs page.
- [quantclaw/state/sessions.py](quantclaw/state/sessions.py) — session boundaries (chat sessions, autopilot sessions, training sessions).
- [quantclaw/state/memory.py](quantclaw/state/memory.py) — cross-session memory indexable by agent.
- [quantclaw/state/observability.py](quantclaw/state/observability.py) — derived metrics (event counts, p95 latencies) for the dashboard.

### 19.3 Playbook vs State DB — when to use which?

| Use the Playbook for | Use the State DB for |
| --- | --- |
| Mission-level decisions and narrative | High-volume structured events |
| Anything a human might read chronologically | Anything the dashboard queries |
| Campaign and deployment snapshots | Task/plan/session records |
| User preferences | Derived metrics |

Rough rule: **the Playbook tells the story, the DB answers questions.**

## 20. Events, Narration, And Notification Routing

A good harness is not just smart. It is also visible. This section covers how.

### 20.1 The event bus

Every interesting thing that happens in QuantClaw becomes a typed `Event(type, payload, source_agent)` on the EventBus. See [quantclaw/events/types.py](quantclaw/events/types.py) for the full catalog (50+ types). Selected highlights:

| Namespace | Examples |
| --- | --- |
| `orchestration.*` | `cycle_complete`, `evaluation`, `campaign_updated`, `allocation_updated`, `step_started`, `step_completed`, `step_failed`, `broadcast` |
| `agent.*` | `task_started`, `task_completed`, `task_failed` |
| `trade.*` | `submitted`, `filled`, `reconciliation_fail` |
| `market.*` | `gap_detected`, `regime_change` |
| `cost.*` | `budget_warning` — now fired by the LLMRouter when cumulative USD spend crosses a configured threshold (Section 10.4) |
| `pipeline.*` | `started`, `stage_complete` |
| `factor.*` | `discovered`, `rejected` |
| `schedule.*` | `triggered` |

Because events are typed (not free-form strings), subscribers can pattern-match reliably. This is what makes Sentinel's rules table possible.

### 20.2 Three consumers

Events flow to three consumers simultaneously:

1. **Sentinel** — pattern-matches against alert rules (Section 17)
2. **EventPersister** — batch-writes to the SQLite `events` table every 5s (Section 19)
3. **Notification sinks** — routes to external channels based on config

### 20.3 Notification routing

Routing rules live in [quantclaw/config/default.yaml](quantclaw/config/default.yaml) under `notification_routes`:

```yaml
notification_routes:
  - event: "market.*"
    channels: [telegram, discord]
    urgency: immediate
  - event: "trade.reconciliation_fail"
    channels: [telegram, discord, slack]
    urgency: immediate
  - event: "agent.task_failed"
    channels: [telegram, discord]
    urgency: high
  - event: "cost.budget_warning"
    channels: [telegram]
    urgency: high
  - event: "pipeline.*"
    channels: [discord]
    urgency: normal
  - event: "factor.*"
    channels: [discord]
    urgency: normal
  - event: "schedule.triggered"
    channels: [discord]
    urgency: low
```

Each rule matches an event pattern, lists the channels to deliver to, and sets an urgency class. Three sinks are implemented:

- [quantclaw/notifications/telegram.py](quantclaw/notifications/telegram.py) — POSTs to `api.telegram.org/bot{token}/sendMessage` with Markdown parse mode
- [quantclaw/notifications/discord.py](quantclaw/notifications/discord.py) — webhook-based
- [quantclaw/notifications/slack.py](quantclaw/notifications/slack.py) — webhook-based

### 20.4 The formatter

[quantclaw/notifications/formatter.py](quantclaw/notifications/formatter.py) renders an event into a human-readable string keyed by urgency: `[URGENT]`, `[ALERT]`, `[INFO]`, `[LOG]` prefix plus the event name and payload lines. This keeps sinks interchangeable — they only ever receive pre-formatted strings.

### 20.5 Why this is the right shape

- **Typed events** make subscription deterministic and testable
- **Config-driven routing** means changing "send failures to Slack too" is a one-line edit, not code
- **Urgency tiers** decouple "what happened" from "how loudly should I say it"
- **Multiple sinks per event** means notification fan-out is free
- **A dedicated persister** keeps event volume off the hot path while preserving everything for later queries
- **Cost events close the loop** — spend now produces real warnings, which route to Telegram as `high`-urgency messages before the budget is exhausted

## 21. The Plugin System: Data Sources, Brokers, Engines

The harness is opinionated about *how* things run, but intentionally agnostic about *what* data it reads or *which* broker it trades through. That flexibility lives in the plugin system.

### 21.1 The manager

See [quantclaw/plugins/manager.py](quantclaw/plugins/manager.py) and [quantclaw/plugins/interfaces.py](quantclaw/plugins/interfaces.py).

Four plugin types:

| Type | Contract | Examples |
| --- | --- | --- |
| `data` | `fetch_ohlcv(symbol, start, end) → DataFrame` | yfinance, FRED, SEC Edgar, Alpha Vantage |
| `broker` | `connect()`, `submit_order()`, `get_positions()`, `get_account()`, `is_market_open()` | broker_ib (Interactive Brokers), paper (built-in) |
| `engine` | `backtest(strategy, data, config) → BacktestResult` | engine_builtin |
| `asset` | asset universe definitions | asset_us_equities |

The `PluginManager` provides `register()`, `get()` (lazy instantiation), `list_plugins()`, `discover()` (via Python entry_points so third parties can ship plugins as wheels), plus `install()` / `uninstall()` that wrap pip.

### 21.2 Builtin data plugins

Inside [quantclaw/plugins/builtin/](quantclaw/plugins/builtin/):

**Free (no API key needed):**
- `data_yfinance` — Yahoo Finance equities/ETFs/crypto
- `data_stooq` — Stooq historical data
- `data_fred` — Federal Reserve Economic Data
- `data_sec_edgar` — SEC filings
- `data_worldbank`, `data_imf`, `data_bis` — macro datasets
- `data_bls` — Bureau of Labor Statistics
- `data_treasury`, `data_ecb` — treasury and ECB rates
- `data_cftc` — CFTC commitments of traders
- `data_openinsider` — insider trading filings

**API-key sources (optional):**
- `data_alphavantage`, `data_finnhub`, `data_fmp`, `data_simfin`, `data_tiingo`, `data_twelvedata`, `data_nasdaq`, `data_eia`

All data plugins are wrapped by `CachedDataPlugin`, which memoizes responses to `data/cache/` to cut down on duplicate API calls.

### 21.3 Activation

Plugins are activated through [config/default.yaml](quantclaw/config/default.yaml) under `plugins:`:

```yaml
plugins:
  broker: broker_ib
  data:
    - data_yfinance
    - data_fred
    - data_sec_edgar
    # ...
  engine: engine_builtin
  asset: [asset_us_equities]
```

The Ingestor agent reads this list and fans out to every active data source in parallel. New plugins plug in by appending one line to config plus a pip install — no code changes to core.

### 21.4 Why this is good harness engineering

- New data sources are **pluggable, not forkable**
- Brokers are **swappable** — the same Executor code runs against paper, IB, or any future broker
- The backtest engine itself is **replaceable**, which matters when you want to try vectorbt or zipline without rewriting the system
- The dependency graph is clean: agents depend on interfaces, interfaces are implemented by plugins, config picks implementations

## 22. Progression, Milestones, And Onboarding

Here is a layer that surprises most beginners: **the harness includes a deliberate human-pacing system**. QuantClaw does not hand a brand-new user the full arsenal. It unlocks capability over time, gated by demonstrated progress.

### 22.1 Seven levels

See [quantclaw/progression/levels.py](quantclaw/progression/levels.py).

| Level | Name | New agents unlocked | New templates |
| --- | --- | --- | --- |
| 0 | Observer | scheduler only | baselines |
| 1 | Paper Trader | executor, ingestor, backtester, evaluator, reporter, debugger | + beginner |
| 2 | Tinkerer | (same — learn to edit parameters) | (same) |
| 3 | Live Trader | risk_monitor, sentinel, compliance | + intermediate |
| 4 | Strategist | researcher | (same) |
| 5 | Quant | miner, trainer | + advanced, + options |
| 6 | Architect | planner | full feature set |

The level is stored in [quantclaw.yaml](quantclaw.yaml) as `level: N` and checked at agent instantiation time by [quantclaw/execution/pool.py](quantclaw/execution/pool.py).

### 22.2 Milestones

See [quantclaw/progression/milestones.py](quantclaw/progression/milestones.py). Level advancement is driven by verifiable facts:

- `first_backtest` — total_backtests ≥ 1
- `three_backtests` — total_backtests ≥ 3
- `beat_spy` — best strategy Sharpe > SPY Sharpe over same window
- `paper_30d` — paper trading days ≥ 30
- `paper_positive` — paper equity positive after 30 days
- `first_live_profit` — any live P&L > 0
- `sharpe_above_1` — live 3-month Sharpe > 1.0
- `multi_strategy` — 3+ active paper deployments simultaneously

Each is a pure function on the system's metrics — no subjective judgement — which makes the unlock path auditable.

### 22.3 Onboarding

See [quantclaw/progression/onboarding.py](quantclaw/progression/onboarding.py). On first run, the dashboard walks a new user through:

- LLM provider selection (OAuth or API key)
- optional API-key data source wiring
- initial watchlist (defaulting to SPY / QQQ)
- level confirmation (defaulting to 3 = Live Trader, overridable)
- writing `quantclaw.yaml` so onboarding does not repeat

### 22.4 Why this is good harness engineering

This is probably the least obvious harness layer. But it is real engineering:

- **Safety by capability gating** — a Level 0 user cannot accidentally unleash the miner on an unfunded account
- **Pedagogical scaffolding** — users learn the system in a sensible order rather than getting dropped into the deep end
- **Code as curriculum** — the `content/` folder ([quantclaw/progression/content/](quantclaw/progression/content/)) holds per-level learning material that the dashboard surfaces

The lesson: **a harness for a powerful system should meter that power out to users as they show they can handle it.**

## 23. The Dashboard Frontend

The dashboard is a Next.js application at [quantclaw/dashboard/app/](quantclaw/dashboard/app/). It is the user's primary window into the harness.

### 23.1 Stack

- **Next.js** (App Router) + **TypeScript** + **Tailwind CSS**
- Internationalization via `lang-context.tsx` (en / zh / ja at minimum)
- Connects to the Python backend over HTTP (on port 8000) and the sidecar (port 24122)

### 23.2 Pages

Under [quantclaw/dashboard/app/app/dashboard/](quantclaw/dashboard/app/app/dashboard/):

| Page | Purpose |
| --- | --- |
| `chat/` | Primary conversational interface — where the user sets goals and asks questions |
| `floor/` | The cyberpunk "trading floor" — live view of active campaigns and deployments |
| `agents/` | Per-agent status, recent tasks, and configuration |
| `strategies/` | Strategy library browser; view, edit, delete, clone |
| `backtest/` | Kick off backtests interactively |
| `portfolio/` | Paper and live portfolio state |
| `risk/` | Drawdown charts, concentration metrics, compliance status |
| `logs/` | Event stream (reads from the `events` table) |
| `learn/` | Progression content from `quantclaw/progression/content/` |
| `settings/` | LLM provider, API keys, data sources, notification channels, theme |

Plus an `onboarding/` flow that runs on first launch.

### 23.3 How it talks to the backend

See [quantclaw/dashboard/api.py](quantclaw/dashboard/api.py) (FastAPI). The backend exposes endpoints for:

- goal submission (`POST /goal`) → hits `OODALoop.set_goal_persistent()`
- plan approval (`POST /plans/{id}/approve`) → hits the plan store and unblocks the dispatcher
- live event subscription (`GET /events/stream`, server-sent events)
- portfolio + deployment snapshots
- settings updates (API keys, data sources, progression level)
- OAuth callback handlers (tied into [quantclaw/dashboard/oauth.py](quantclaw/dashboard/oauth.py))

### 23.4 Why it matters to the harness

A harness without a UI still works, but a harness *with* a well-designed UI becomes trustworthy. The dashboard lets a human:

- see what agents are doing right now (observability)
- approve or reject plans before they execute (control)
- inspect why a decision was made (auditability)
- pause / cancel mid-cycle (safety)

That is the human side of the harness. Everything else in this notebook is how the *system* stays in control; the dashboard is how the *user* stays in control.

## 24. Safety And Risk Boundaries

This is a critical beginner lesson: good harness engineering is not just about making the system more autonomous. It is also about making it safer.

QuantClaw enforces safety at **every layer**, not just at trade time:

### 24.1 Mission-level

- Broad goals are compiled into campaigns, not direct live trade commands (Section 5)
- The campaign is explicitly paper-first; phase transitions require evidence (Section 6)
- Planner rules forbid suggesting live trading during the paper phase (Section 9)

### 24.2 Execution-level

- The executor still checks trust level before using a real broker (Section 16)
- Compliance checks run on every order batch (Section 8, Compliance agent)
- RiskMonitor is a daemon continuously watching drawdown, sector concentration, and leverage (Section 8, 17)

### 24.3 Code-execution level

- LLM-generated factor code runs in a **subprocess sandbox** with **AST-enforced** import whitelist, blocked builtins (`exec`, `eval`, `getattr`, etc.), blocked dunder attribute access (`__class__`, `__subclasses__`, etc.), resource caps, and environment-variable stripping (Section 13)
- Non-allowed imports, blocked builtins, and dunder accesses are **critical** — execution is rejected before the subprocess is spawned, not merely warned about
- Strategy modules are validated to conform to the `Strategy` class contract before loading (Section 14)
- `signals()` / `allocate()` return values are checked for shape and finiteness before the executor acts on them

### 24.4 User-level

- Progression levels gate which agents and templates are available (Section 22)
- Plans require approval in the dashboard unless the user opted into autopilot (Section 11)
- The dashboard stop button actually halts mid-plan via the `_cancel` event (Section 11)

### 24.5 Observability-level

- Sentinel fires alerts on failure streaks, cost spikes, and regime changes (Section 17)
- The LLMRouter fires `cost.budget_warning` the first time cumulative USD spend crosses a configured threshold (Section 10.4, 20)
- Notification routing makes trade failures *immediate* across all three channels (Section 20)
- Every event is persisted to the State DB for forensic analysis (Section 19)

### 24.6 Supply-chain level

- CI runs `pip-audit --strict` on every push, so upstream CVEs in declared dependencies fail the build before they reach main ([.github/workflows/ci.yml](.github/workflows/ci.yml))
- OAuth secrets are read from env with public-CLI fallbacks; the private `.env` and `data/oauth_credentials.json` are gitignored

So `"go make me cash"` does **not** mean:

- skip validation
- skip paper trading
- skip compliance
- skip trust gates
- skip resource limits on factor code
- skip AST validation of factor code
- skip human approval of plans
- skip notification on failures
- skip notification on budget spend
- skip dependency auditing

It means:

- start a persistent profit-seeking research-and-paper-trading campaign inside guardrails

That distinction is exactly what a good harness is supposed to enforce.

## 25. End-To-End Walkthrough: `go make me cash`

Now let us walk the whole path end to end.

### Step 1: The goal arrives

The user types `go make me cash` into the dashboard chat page ([quantclaw/dashboard/app/](quantclaw/dashboard/app/), Section 23). The frontend POSTs `/goal`, which hits [OODALoop.set_goal_persistent()](quantclaw/orchestration/ooda.py).

### Step 2: The campaign activates

`CampaignManager.activate()` (Section 5) recognizes this as a broad goal pattern from [config/default.yaml](quantclaw/config/default.yaml) and creates a `ProfitCampaign` in `discover` phase.

### Step 3: Autopilot turns on

If campaign auto-activation is enabled, the autonomy mode is pushed into `AUTOPILOT`.

### Step 4: The campaign is persisted

The system writes `campaign_state` and `ceo_preference` entries to the Playbook (Section 18).

### Step 5: OODA observes current context

The loop reads recent playbook entries, trust state, autonomy mode, current deployment allocator context, and recent events from the State DB.

### Step 6: OODA orients around the next subgoal

Instead of using the raw goal every time, it asks the campaign manager for the next phase-specific subgoal.

Early on that subgoal sounds like:

- explore for materially different profitable strategies
- favor breadth and novelty
- paper trade only after validation

### Step 7: The planner builds a full workflow

The Planner (Section 9) receives campaign context, allocator context, the agent manifest, and playbook history. It emits a `Plan` with PlanSteps like:

- `researcher` → find candidate factor families
- `ingestor` (depends_on: researcher) → fetch OHLCV for candidate universe
- `miner` (depends_on: ingestor) → generate factor hypotheses, run in sandbox
- `trainer` (depends_on: miner) → train predictive model on top factors
- `backtester` (depends_on: trainer) → backtest the resulting strategy
- `evaluator` (depends_on: backtester) → held-out validation
- `reporter` (depends_on: evaluator) → summarize

### Step 8: The dispatcher runs the DAG

The Dispatcher (Section 11) walks the DAG, parallelizes what it can, injects upstream results into downstream tasks, and emits step-level events throughout.

### Step 9: Sandboxed code runs

Miner and Trainer produce Python code that runs in the sandbox (Section 13) with timeout, memory, and AST protections. Their outputs include concrete `Strategy` class files saved under `data/strategies/`.

### Step 10: The cycle gets evaluated

The system records metrics like Sharpe, held-out Sharpe, verdict, and whether paper execution happened. These are persisted to both the Playbook and the State DB.

### Step 11: The campaign phase may shift

If discovery is strong enough, the campaign moves to `validate`. If held-out evidence is strong enough, it moves to `paper` (Section 6).

### Step 12: The allocator harvests the best candidates

`DeploymentAllocator.rebalance()` (Section 15) extracts a candidate from cycle results, scores it, and decides whether it belongs in the active set or watchlist. A new `deployment_state` and `allocation_decision` are appended to the Playbook.

### Step 13: Paper phase becomes deterministic

Once there are active paper deployments, `OODALoop.decide()` bypasses normal free-form planning and builds a deterministic plan (Section 16):

- step 0: executor runs active deployments (`run_deployments`)
- step 1: reporter summarizes paper results

### Step 14: The executor runs the paper portfolio

It loads strategies via the loader (Section 14), fetches data via plugins (Section 21), computes target weights, aggregates them, and submits paper rebalance orders.

### Step 15: Sentinel watches the whole time

Throughout every step, Sentinel (Section 17) is listening. If any agent fails three cycles in a row, or drawdown hits 80% of the limit, Sentinel fires alerts — which Telegram and Discord receive within seconds (Section 20).

### Step 16: The allocator updates again

Based on fresh paper outcomes and new research, the allocator can keep, add, downweight, or retire strategies. `allocation_updated` events flow out.

### Step 17: The loop keeps going

`run_continuous()` no longer hard-stops broad campaigns after a few cycles. It keeps checkpointing and iterating. If the process restarts, `restore_persistent_state()` rebuilds the campaign and allocator from the Playbook and the system picks up where it left off.

That is the full closed-loop harness.

## 26. Why This Is Good Harness Engineering

Let us name the strongest design decisions directly.

### 1. Broad goals are compiled, not obeyed literally
This prevents unsafe or naive behavior and turns plain language into a manageable mission structure.

### 2. The outer loop and inner loop are separated
Campaign logic handles mission progression. OODA handles cycle execution. This keeps responsibilities clean.

### 3. State is externalized
Campaigns and deployments live in the Playbook; tasks, events, and plans live in the State DB. Not in model context.

### 4. Deterministic runners are used where appropriate
Paper portfolio execution is too important to leave entirely to free-form replanning. The `run_deployments` path is hand-written.

### 5. Portfolio management is first-class
The Allocator manages multiple paper incumbents and challengers instead of chasing a single winner.

### 6. Safety is a layered architecture, not a layer
Paper-first campaigns, sandbox isolation, compliance gates, risk daemon, Sentinel alerts, progression unlocks — safety is present at every altitude.

### 7. Providers are abstracted
The LLMRouter + sidecar means agents are LLM-agnostic. Swap providers in config, not code.

### 8. Agents are specialists with a manifest
Every agent has a single, named job with declared inputs, outputs, and dependencies. No god-agents.

### 9. LLM-generated code is a first-class concern
The sandbox exists specifically because "run the code the LLM wrote" deserves its own subsystem with its own guarantees.

### 10. The harness is observable and testable
Typed events, playbook entries, notification routes, and regression tests make the behavior inspectable. The dashboard surfaces all of it.

### 11. The user is part of the harness
Progression gates capability, plans require approval, the stop button works. Humans are not an afterthought.

### 12. Extensibility is enforced by contracts
Plugins implement narrow interfaces; strategies implement a narrow contract; agents have declared schemas. Adding something new is a matter of filling in a contract, not rewriting the core.

These are the kinds of design choices that separate a fun demo from a serious agentic system.

## 27. Tests As Executable Documentation

A very good way to learn a harness is to read its tests. They tell you what behavior the authors consider load-bearing enough to defend.

The most useful tests for the campaign / allocator / executor stack:

- [tests/test_campaigns.py](tests/test_campaigns.py)
- [tests/test_executor.py](tests/test_executor.py)

What they prove:

- a broad goal like `go make me cash` starts a campaign
- campaigns restore correctly from persisted state
- phase promotion works from discover → validate → paper
- the allocator produces active and watchlist sets with normalized weights
- OODA includes deployment context during observe and orient
- paper phase builds a deterministic deployment runner plan
- the executor can actually run paper deployments end to end

Then widen out to get coverage of the rest of the codebase:

- [tests/](tests/) — browse for `test_sandbox.py`, `test_router.py`, `test_dispatcher.py`, `test_plugins.py`, `test_progression.py`, etc.

That is exactly what good tests should do: protect the intended behavior of the harness and teach new contributors what the system promises.

## 28. Practical Study Order

If you want to learn this codebase deeply, read it in this order. The order follows the architecture from the outside in, then out again to extensibility surfaces.

**Outer mission layer:**
1. [quantclaw/orchestration/campaigns.py](quantclaw/orchestration/campaigns.py)
2. [quantclaw/orchestration/deployments.py](quantclaw/orchestration/deployments.py)
3. [quantclaw/orchestration/ooda.py](quantclaw/orchestration/ooda.py)
4. [quantclaw/orchestration/playbook.py](quantclaw/orchestration/playbook.py)

**Planning and execution:**
5. [quantclaw/execution/planner.py](quantclaw/execution/planner.py)
6. [quantclaw/execution/plan.py](quantclaw/execution/plan.py)
7. [quantclaw/execution/dispatcher.py](quantclaw/execution/dispatcher.py)
8. [quantclaw/execution/router.py](quantclaw/execution/router.py)
9. [quantclaw/execution/tool_loop.py](quantclaw/execution/tool_loop.py)

**Agents (in manifest order):**
10. [quantclaw/agents/base.py](quantclaw/agents/base.py)
11. [quantclaw/agents/manifest.py](quantclaw/agents/manifest.py)
12. [quantclaw/agents/executor.py](quantclaw/agents/executor.py)
13. [quantclaw/agents/sentinel.py](quantclaw/agents/sentinel.py)
14. [quantclaw/agents/miner.py](quantclaw/agents/miner.py), [trainer.py](quantclaw/agents/trainer.py), [backtester.py](quantclaw/agents/backtester.py), [evaluator.py](quantclaw/agents/evaluator.py)

**Sandbox and strategy:**
15. [quantclaw/sandbox/sandbox.py](quantclaw/sandbox/sandbox.py)
16. [quantclaw/sandbox/security.py](quantclaw/sandbox/security.py)
17. [quantclaw/sandbox/model_trainer.py](quantclaw/sandbox/model_trainer.py)
18. [quantclaw/strategy/loader.py](quantclaw/strategy/loader.py)
19. [quantclaw/strategy/generator.py](quantclaw/strategy/generator.py)
20. [quantclaw/strategy/templates/](quantclaw/strategy/templates/)

**State and observability:**
21. [quantclaw/state/db.py](quantclaw/state/db.py)
22. [quantclaw/state/event_persister.py](quantclaw/state/event_persister.py)
23. [quantclaw/events/types.py](quantclaw/events/types.py)
24. [quantclaw/notifications/](quantclaw/notifications/)

**Extensibility:**
25. [quantclaw/plugins/manager.py](quantclaw/plugins/manager.py)
26. [quantclaw/plugins/interfaces.py](quantclaw/plugins/interfaces.py)
27. [quantclaw/plugins/builtin/](quantclaw/plugins/builtin/)

**Sidecar and frontend:**
28. [quantclaw/sidecar/server.js](quantclaw/sidecar/server.js)
29. [quantclaw/dashboard/api.py](quantclaw/dashboard/api.py)
30. [quantclaw/dashboard/oauth.py](quantclaw/dashboard/oauth.py)
31. [quantclaw/dashboard/app/](quantclaw/dashboard/app/) — skim the Next.js structure

**Progression:**
32. [quantclaw/progression/levels.py](quantclaw/progression/levels.py)
33. [quantclaw/progression/milestones.py](quantclaw/progression/milestones.py)

**Configuration and tests:**
34. [quantclaw/config/default.yaml](quantclaw/config/default.yaml)
35. [tests/test_campaigns.py](tests/test_campaigns.py)
36. [tests/test_executor.py](tests/test_executor.py)

Why this order works:

- you first learn the mission harness (what the system *wants*)
- then the planning and dispatch machinery (how it *decides and acts*)
- then the agents (who *does* the work)
- then the sandbox + strategy contract (how *unsafe code becomes safe capability*)
- then state and observability (how it *remembers and tells us*)
- then extensibility (how *new capabilities enter the system*)
- then the human surfaces (how *people interact with it*)
- then progression (how *capability is metered out*)
- finally config and tests (the *knobs and the proofs*)

## 29. Optional Hands-On Cells

The next few cells are optional. They help you inspect the real code and state locally.

In [ ]:
# Optional hands-on: find the main harness surfaces quickly.
!rg -n "class CampaignManager|class DeploymentAllocator|class LLMRouter|class Sandbox|class SentinelAgent|run_deployments|_build_paper_deployment_plan|restore_persistent_state" quantclaw/orchestration quantclaw/execution quantclaw/agents quantclaw/sandbox tests


In [ ]:
# Optional hands-on: enumerate every agent by walking the manifest.
from quantclaw.agents.manifest import get_manifest
for name, spec in get_manifest().items():
    print(f"{name:12s}  llm={spec.uses_llm!s:5}  daemon={spec.daemon!s:5}  {spec.description}")


In [ ]:
# Optional hands-on: inspect recent persisted campaign and deployment state.
import json
from pathlib import Path

path = Path("data/playbook.jsonl")
if path.exists():
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    interesting = [r for r in rows if r.get("entry_type") in {"campaign_state", "deployment_state", "allocation_decision"}]
    for row in interesting[-10:]:
        print(row["entry_type"], row["content"])
else:
    print("No playbook file found yet.")


In [ ]:
# Optional hands-on: peek into the State DB to see what's been happening.
import sqlite3
from pathlib import Path

db_path = Path("data/quantclaw.db")
if db_path.exists():
    with sqlite3.connect(db_path) as conn:
        for table in ("events", "tasks", "plans", "sessions"):
            try:
                cur = conn.execute(f"SELECT COUNT(*) FROM {table}")
                print(f"{table:10s} rows: {cur.fetchone()[0]}")
            except sqlite3.OperationalError:
                print(f"{table:10s} (not yet created)")
        print("\nRecent events:")
        for row in conn.execute("SELECT event_type, source_agent, created_at FROM events ORDER BY created_at DESC LIMIT 10"):
            print("  ", row)
else:
    print("No state DB yet - run QuantClaw once to create it.")


## 30. Final Takeaway

If you remember only one thing, remember this:

QuantClaw's harness no longer treats `"go make me cash"` as a single request to answer. It treats it as a standing mission to manage — and that mission is surrounded by **eight cooperating layers**:

1. **Mission harness** — CampaignManager compiles broad goals into persistent state machines
2. **Orchestration harness** — OODA + Planner + Dispatcher + LLMRouter + Sidecar
3. **Agent harness** — twelve specialists with a manifest, a base class, and declared dependencies
4. **Execution harness** — Sandbox for LLM code, Strategy contract for portable modules, Executor for paper deployments
5. **Portfolio harness** — DeploymentAllocator with scoring, qualification, and normalized weights
6. **Memory harness** — append-only Playbook for story, SQLite State DB for queries
7. **Observability harness** — typed Events, EventPersister, Notification sinks with urgency routing, Sentinel reacting to it all
8. **Human harness** — Progression levels, plan approvals, the dashboard stop button, onboarding

No one of these layers is the system. The system is the **composition**.

That is what excellent harness engineering looks like in practice: not one clever prompt, not one clever agent, not even one clever architecture diagram — but a coordinated stack of layers, each doing a job small enough to reason about, together forming something that can keep exploring, validating, allocating, and executing over time, *safely*, for as long as a user wants it to.

The next time someone says "the model is the product," remember QuantClaw. The model is the clever worker inside the building. The harness is the building itself — the walls, the wiring, the doors, the security system, the elevator, the lobby, and the fire alarm. You cannot ship the worker without the building.